##### Imports and constants

In [ ]:
import azure.functions as func
import logging
import json
import urllib.request
import pandas as pd

import re

In [4]:
ENDPOINT = "https://inpro-participants-submit-3avzbu5vaa-as.a.run.app"
#change mappings according to the final form (change left side)
mapping = { "Telegram handle": "tele_handle",
           "Full name": "full_name",
            "Student email": "student_email",
            "Trail 1": "trail_first_choice",
            "Trail 2": "trail_second_choice",
            "Trail 3": "trail_third_choice"}

##### Function for sending https req to firebase

In [ ]:
def upload_data(payload : dict): #function to upload data onto firebase (DONT NEED TOUCH)
    data = json.dumps(payload).encode()
    req = urllib.request.Request(
        ENDPOINT,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req) as resp:
            body = resp.read().decode()
            print(f"Status: {resp.status}")
            print(f"Response: {body}")
    except urllib.error.HTTPError as e:
        body = e.read().decode()
        print(f"Status: {e.code}")
        print(f"Error: {body}")

##### Trial Data

In [7]:
df = pd.read_excel("C:\\Users\\xuan2\\Downloads\\responses.xlsx").rename(columns=mapping)


##### Function to read 1 row and process accordingly to make it same format as payload


In [8]:
def parse_numeric_from_string(v):
        if v is None:
            return None
        if isinstance(v, (int, float)) and not pd.isna(v):
            try:
                return int(v)
            except Exception:
                pass
        s = str(v)
        m = re.search(r'(\d+)', s)
        return int(m.group(1)) if m else s


def format_vals(init: dict) -> dict:
    if 'year' in init:
        init['year'] = parse_numeric_from_string(init['year'])
    for k in ('trail_first_choice', 'trail_second_choice', 'trail_third_choice'):
        if k in init:
            init[k] = parse_numeric_from_string(init[k])
    return init


def process_row(row: pd.DataFrame) -> dict:
    srs = row.squeeze()
    cols = ['tele_handle', 'full_name', 'house', 'student_email', 'year', 'status', 'role', 'trail_first_choice', 'trail_second_choice', 'trail_third_choice'] #edit col names according to the final one
    d = {}
    for col_name in cols:
        if col_name == 'role':
            d['role'] = "Participant"
        if col_name == 'status':
            d['status'] = 'pending'
        else:
            val = srs[col_name]
            d[col_name] = val
    return format_vals(d)

In [9]:
read = df.tail(1)
process_row(read)

KeyError: 'tele_handle'